# math

> Pure math functions for scrollbar thumb position and size calculation.

In [ ]:
#| default_exp core.math

In [ ]:
#| export
from typing import Optional, Tuple

In [ ]:
#| export
def compute_scrollbar(position: int,           # Current position (0-indexed)
                      visible_count: int,       # Number of visible items
                      total_items: int,          # Total item count
                      track_height: float,       # Scrollbar track height in px
                      min_thumb_height: int = 24,  # Minimum thumb height in px
                      max_position: Optional[int] = None,  # Upper bound of position range (None = total_items - visible_count)
                      thumb_ratio: Optional[float] = None,  # Thumb height as fraction of track (None = visible_count / total_items)
                     ) -> Tuple[float, float]:   # (thumb_top_percent, thumb_height_percent)
    """Compute scrollbar thumb position and size as percentages.

    When thumb_ratio and max_position are explicitly provided (cursor-based
    model), the thumb size and range are fully determined by those values
    regardless of the visible_count vs total_items relationship.
    """
    if total_items <= 0:
        return (0.0, 100.0)  # No items — thumb fills track

    # In default (window) mode, all items visible means nothing to scroll
    if thumb_ratio is None and max_position is None and visible_count >= total_items:
        return (0.0, 100.0)

    # Thumb height: use explicit ratio if provided, else visible/total
    if thumb_ratio is not None:
        thumb_height_pct = thumb_ratio * 100.0
    else:
        thumb_height_pct = (visible_count / total_items) * 100.0

    # Enforce minimum thumb height
    if track_height > 0:
        min_pct = (min_thumb_height / track_height) * 100.0
        thumb_height_pct = max(thumb_height_pct, min_pct)

    # Thumb position: proportion of position within scrollable range
    max_pos = max_position if max_position is not None else (total_items - visible_count)
    max_pos = max(1, max_pos)
    thumb_top_pct = (position / max_pos) * (100.0 - thumb_height_pct)

    return (thumb_top_pct, thumb_height_pct)

## Tests

In [ ]:
# Default (window) mode: all items visible → thumb fills track
assert compute_scrollbar(0, 10, 5, 600.0) == (0.0, 100.0)
assert compute_scrollbar(0, 10, 10, 600.0) == (0.0, 100.0)
assert compute_scrollbar(0, 1, 0, 600.0) == (0.0, 100.0)

# Cursor-based mode: all items visible but explicit thumb_ratio/max_position
# → thumb uses the ratio, NOT 100%
top, height = compute_scrollbar(0, 15, 10, 600.0, max_position=9, thumb_ratio=1/10)
assert abs(height - 10.0) < 0.01  # 1/10 = 10%, not 100%
assert top == 0.0

# Cursor at middle when all visible
top, height = compute_scrollbar(5, 15, 10, 600.0, max_position=9, thumb_ratio=1/10)
expected_top = (5 / 9) * (100.0 - height)
assert abs(top - expected_top) < 0.01
print("cursor-based mode (all items visible) tests passed")

In [ ]:
# Position at start → thumb at top (min thumb height may clamp upward)
top, height = compute_scrollbar(0, 15, 500, 600.0)
assert top == 0.0
natural_pct = (15 / 500) * 100.0  # 3%
min_pct = (24 / 600.0) * 100.0    # 4%
assert height == max(natural_pct, min_pct)  # Clamped to min

In [ ]:
# Position at end → thumb at bottom
top, height = compute_scrollbar(485, 15, 500, 600.0)
assert abs(top - (100.0 - height)) < 0.01  # Thumb bottom aligns with track bottom

In [ ]:
# Min thumb height enforcement
top, height = compute_scrollbar(0, 1, 10000, 600.0, min_thumb_height=24)
min_pct = (24 / 600.0) * 100.0  # 4%
assert height == min_pct  # Natural height (0.01%) is below min

In [ ]:
# Mid-position
top, height = compute_scrollbar(250, 15, 500, 600.0)
max_start = 500 - 15
expected_top = (250 / max_start) * (100.0 - height)
assert abs(top - expected_top) < 0.01

In [ ]:
# Card-stack model: max_position = total_items - 1
# Position at last item → thumb at bottom
top, height = compute_scrollbar(499, 3, 500, 600.0, max_position=499)
assert abs(top - (100.0 - height)) < 0.01  # Thumb at bottom

# Position at start → thumb at top
top, height = compute_scrollbar(0, 3, 500, 600.0, max_position=499)
assert top == 0.0

# Mid-position with card-stack model
top, height = compute_scrollbar(250, 3, 500, 600.0, max_position=499)
expected_top = (250 / 499) * (100.0 - height)
assert abs(top - expected_top) < 0.01

# Compare: without max_position, same position overshoots
top_default, _ = compute_scrollbar(250, 3, 500, 600.0)  # max = 500-3 = 497
top_explicit, _ = compute_scrollbar(250, 3, 500, 600.0, max_position=499)
assert top_default != top_explicit  # Different because max ranges differ

print("max_position (card-stack model) tests passed")

In [ ]:
# thumb_ratio override
# Card stack with 25 items, visible_count=7 → default thumb = 28%
# With thumb_ratio = 1/25 = 4%, thumb is much thinner
_, height_default = compute_scrollbar(0, 7, 25, 600.0)
assert abs(height_default - 28.0) < 0.01

_, height_ratio = compute_scrollbar(0, 7, 25, 600.0, thumb_ratio=1/25)
assert abs(height_ratio - 4.0) < 0.01

# min_thumb_height still enforced with thumb_ratio
_, height_tiny = compute_scrollbar(0, 7, 25, 600.0, thumb_ratio=0.001)
min_pct = (24 / 600.0) * 100.0
assert height_tiny == min_pct  # Clamped to min

# thumb_ratio=None behaves same as default
top1, h1 = compute_scrollbar(10, 7, 25, 600.0, thumb_ratio=None)
top2, h2 = compute_scrollbar(10, 7, 25, 600.0)
assert top1 == top2 and h1 == h2

print("thumb_ratio tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()